# 06 — Circuit Analysis

A *circuit* is the minimal set of components sufficient for a behaviour.
We ablate components (zero and mean ablation) and run a greedy search for
the smallest set that keeps task performance above a threshold.

In [ ]:
import sys
sys.path.insert(0, "..")  # run from the notebooks/ directory

import matplotlib
import torch

import kamui
from kamui.model.config import ModelConfig

torch.manual_seed(0)
print("KAMUI", kamui.__version__)

In [ ]:
# Train a tiny model on a synthetic corpus (~30s on CPU).
# The corpus is a seeded word-salad: repetitive enough to learn, varied enough
# that BPE cannot collapse it into a handful of giant tokens.
import random

from kamui.tokenizer.bpe import BPETokenizer
from kamui.training import DataLoader, TextDataset, Trainer, TrainingConfig

rng = random.Random(0)
WORDS = ["the", "cat", "dog", "sat", "ran", "on", "to", "mat", "log", "sun"]
CORPUS = " ".join(rng.choice(WORDS) for _ in range(4000))

config = ModelConfig(n_layers=2, d_model=64, n_heads=4, d_ff=128,
                     vocab_size=300, context_length=32, dropout=0.0)
tokenizer = BPETokenizer.train(CORPUS, vocab_size=config.vocab_size)
tokens = tokenizer.encode(CORPUS)

model = kamui.KAMUITransformer(config)
trainer = Trainer(
    model,
    DataLoader(TextDataset(tokens, config.context_length), batch_size=8, seed=0),
    config=TrainingConfig(max_lr=3e-3, warmup_steps=10, max_steps=1000),
)
records = trainer.train(150)
model.eval()
print(f"loss: {records[0]['train_loss']:.3f} -> {records[-1]['train_loss']:.3f}")

In [ ]:
from kamui.mechinterp import CircuitAblator, find_minimal_circuit

ids = torch.tensor(tokenizer.encode("the cat sat on the"))
target = tokenizer.encode(" mat")[0]

def metric(logits: torch.Tensor) -> float:
    return logits[0, -1, target].item()

ablator = CircuitAblator(model)
for component in ["blocks.0.attn.output", "blocks.1.attn.output", "blocks.1.ffn.output"]:
    print(component, "delta:", round(ablator.ablate([component], ids, metric).delta, 3))

In [ ]:
# Mean ablation is gentler: it removes task-specific signal, not all signal.
result = ablator.mean_ablate(["blocks.1.attn.output"], ids, ids, metric)
print("mean-ablation delta:", round(result.delta, 3))

In [ ]:
# Greedy minimal-circuit search: which components can we NOT remove?
circuit = find_minimal_circuit(model, ids, metric, threshold=0.9)
print("minimal circuit:", circuit)